In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging
import sys

from pathlib import Path
project_root = Path().resolve().parent
sys.path.append(str(project_root))

import os
from collections.abc import Callable
from dataclasses import asdict
from pathlib import Path
from typing import Any

import torch

from experiments.plotting import plot_training_curves
from GNN.parameter_search.helpers import objective_GNN, objective_NN
from GNN.physics_aware_NN import GNN, NN, Regressor
from GNN.training.datasets import build_loaders, build_loaders_NN
from GNN.training.train import build_loss, train_model
from GNN.training.train_config import TrainConfig
from GNN.training.utils import (
    collect_dataset_paths,
    collect_dataset_indices,
    evaluate_loss,
    evaluate_r2,
)
from src.utils import configure_logger

In [3]:
logger = logging.getLogger(__name__)
configure_logger(logging.INFO, logging.INFO)

True

In [4]:
default_model_hparams = {
    "gnn_hidden": 64,
    "gnn_heads": 2,
    "global_hidden": 32,
    "reg_hidden": 128,
    "num_layers": 3,
    "dropout_rate": 0.10,
}

default_train_hparams = {
    "weight_decay": 3e-3,
    "grad_clip": 1.0,
    "early_stopping_patience": 15,
    "early_stopping_min_delta": 0.0,
    "num_workers": 0,
}

PARAMS = {
    "random" : {
        "lr" : 0.0009860340204413903,
        "batch_size" : 128,
        "model_hparams": {
            "gnn_hidden" : 64,
            "gnn_heads": 2,
            "global_hidden": 32,
            "reg_hidden": 128,
            "num_layers": 6,
            "dropout_rate": 0.06640302989664926,
        },
        "train_params": {
            "weight_decay": 3.0500428108369453e-05,
            "grad_clip": 4.379744711312854,
            "early_stopping_patience": 15,
            "early_stopping_min_delta": 0.0,
            "num_workers": 0,
        },
    },
    "clifford" : {
        "lr" : 2.5223176427539664e-05,
        "batch_size" : 32,
        "model_hparams": {
            "gnn_hidden" : 128,
            "gnn_heads": 2,
            "global_hidden": 16,
            "reg_hidden": 32,
            "num_layers": 6,
            "dropout_rate": 0.014842423657881243,
        },
        "train_params": {
            "weight_decay": 1.560010639264171e-05,
            "grad_clip": 7.2440146231033875,
            "early_stopping_patience": 15,
            "early_stopping_min_delta": 0.0,
            "num_workers": 0,
        },
    },
    "haar" : {
        "lr" : 0.0005456850011484297,
        "batch_size" : 16,
        "model_hparams": {
            "gnn_hidden" : 64,
            "gnn_heads": 2,
            "global_hidden": 128,
            "reg_hidden": 16,
            "num_layers": 2,
            "dropout_rate": 0.004127592869557634,
        },
        "train_params": {
            "weight_decay": 1.0439900428164368e-05,
            "grad_clip":  0.011242628935673588,
            "early_stopping_patience": 15,
            "early_stopping_min_delta": 0.0,
            "num_workers": 0,
        },
    },
    "quansistor" : {
        "lr" : 1.0600038808287122e-05,
        "batch_size" : 32,
        "model_hparams": {
            "gnn_hidden" : 128,
            "gnn_heads": 8,
            "global_hidden": 16,
            "reg_hidden": 64,
            "num_layers": 3,
            "dropout_rate": 0.4050959484660077,
        },
        "train_params": {
            "weight_decay": 0.0006626449388616718,
            "grad_clip":  4.779019014790367,
            "early_stopping_patience": 15,
            "early_stopping_min_delta": 0.0,
            "num_workers": 0,
        },
    },
}


MODEL_REGISTRY = {
    "gnn": {
        "build_loaders": build_loaders,
        "build_model": lambda node_in_dim, global_in_dim, hparams: GNN(
            node_in_dim=node_in_dim,
            global_in_dim=global_in_dim,
            gnn_hidden=hparams.get("gnn_hidden", 32),
            gnn_heads=hparams.get("gnn_heads", 8),
            global_hidden=hparams.get("global_hidden", 16),
            reg_hidden=hparams.get("reg_hidden", 16),
            num_layers=hparams.get("num_layers", 5),
            dropout_rate=hparams.get("dropout_rate", 0.1),
        ),
        "returns_nodes_dim": True,
        "objective_fn": objective_GNN,
    },
    "nn": {
        "build_loaders": build_loaders_NN,
        "build_model": lambda node_in_dim, global_in_dim, hparams: NN(
            in_dim=global_in_dim,
            hidden_dim=hparams.get("hidden_dim", 64),
            dropout_rate=hparams.get("dropout_rate", 0.1),
        ),
        "returns_nodes_dim": False,
        "objective_fn": objective_NN,
    },
    "regressor": {
        "build_loaders": build_loaders_NN,
        "build_model": lambda node_in_dim, global_in_dim, hparams: Regressor(
            in_dim=global_in_dim,
            hidden_dim=hparams.get("hidden_dim", 64),
            dropout_rate=hparams.get("dropout_rate", 0.1),
        ),
        "returns_nodes_dim": False,
    },
}

In [5]:
model_type="gnn"
epochs = 15
lr = 1.5131621801102364e-05
model_hparams=None
train_hparams=None
loss_type = "huber"   # "mse" | "huber"
batch_size = 16
training_mode = "per_family"  # "global" | "per_family"
family = "random"  # required if training_mode == "per_family"
target = "sre"
training_data_dir = "final/data/saturated"
# training_data_dir = "../outputs/data/datasets_SRE"
target_variant = "sre_density"  # "sre" | "sre_density" | "log_sre" | "sqrt_sre"
model_save_path = f"../notebooks/outputs/models/final/{family}_model_{model_type}_{training_mode}_{target_variant}.pt"
show_progress=True
show_val_progress=False
log_every_n_batches=10
heartbeat_secs=60.0
epoch_time_warning_secs=600.0
training_scope = "family" if training_mode == "per_family" else "global"
plot_qubits = 6
plot_layers = 100

split = "target"  # "target" | "family"

In [6]:
lr = PARAMS.get(family, {}).get("lr", lr)
batch_size = PARAMS.get(family, {}).get("batch_size", batch_size)
model_hparams = PARAMS.get(family, {}).get("model_hparams", default_model_hparams) if model_hparams is None else default_model_hparams
train_hparams = PARAMS.get(family, {}).get("train_params", default_train_hparams) if train_hparams is None else default_train_hparams

In [7]:
VALID_FAMILIES = {"haar", "clifford", "quansistor", "random"}
if model_type not in MODEL_REGISTRY:
    msg = f"Unsupported model_type: {model_type}. Must be one of {sorted(MODEL_REGISTRY)}"
    raise ValueError(msg)

if training_mode not in {"global", "per_family"}:
    msg = "training_mode must be 'global' or 'per_family'"
    raise ValueError(msg)

if training_mode == "per_family":
    if family is None:
        raise ValueError("family must be provided when training_mode='per_family'")
    if family not in VALID_FAMILIES:
        msg = f"Invalid family: {family}. Must be one of {sorted(VALID_FAMILIES)}"
        raise ValueError(
            msg,
        )

logger.info(f"Starting training | model_type={model_type} | training_mode={training_mode} | family={family} | loss_type={loss_type}")
cfg = TrainConfig(
    epochs=epochs,
    lr=lr,
    loss_type=loss_type,
    batch_size=batch_size,
    training_mode=training_mode,
    family=family,
    target=target,
    show_progress=show_progress,
    show_val_progress=show_val_progress,
    log_batch_loss_every=log_every_n_batches,
    heartbeat=heartbeat_secs,
    epoch_warning=epoch_time_warning_secs,
)
logger.info("Training configuration done.")

model_hparams = {} if model_hparams is None else dict(model_hparams)
train_hparams = {} if train_hparams is None else dict(train_hparams)

2026-06-25 09:30:00,790 - __main__ - INFO - Starting training | model_type=gnn | training_mode=per_family | family=random | loss_type=huber
2026-06-25 09:30:00,795 - __main__ - INFO - Training configuration done.


In [8]:
family_filter = family if training_mode == "per_family" else None
family_projection = family if training_mode == "per_family" else None
num_workers = int(train_hparams.get("num_workers", 0))
logger.info("Collecting data paths...")
train_paths = collect_dataset_indices(
    training_data_dir,
    family=family_filter,
)
if not train_paths:
    raise RuntimeError("No data paths found.")
logger.info(f"Found {len(train_paths)} data paths.")
logger.info("Data paths collected.")

2026-06-25 09:30:01,190 - __main__ - INFO - Collecting data paths...
2026-06-25 09:30:01,192 - __main__ - INFO - Found 1 data paths.
2026-06-25 09:30:01,193 - __main__ - INFO - Data paths collected.


In [9]:
spec = MODEL_REGISTRY[model_type]
logger.info(f"Building loaders and model for model_type={model_type}...")
Loader = Callable[..., Any]

loader_fn: Loader = spec["build_loaders"]
returns_nodes_dim: bool = spec.get("returns_nodes_dim", False)
if returns_nodes_dim:
    train_loader, val_loader, test_loader, node_in_dim, global_in_dim, base_dataset = loader_fn(
        train_paths,
        batch_size=cfg.batch_size,
        seed=cfg.seed,
        train_split=cfg.train_split,
        val_split=cfg.val_split,
        family_projection=family_projection,
        target_variant=target_variant,
        split = split,
        num_workers = num_workers,
    )
else:
    train_loader, val_loader, test_loader, global_in_dim, base_dataset = loader_fn(
        train_paths,
        batch_size=cfg.batch_size,
        seed=cfg.seed,
        train_split=cfg.train_split,
        val_split=cfg.val_split,
        family_projection=family_projection,
        target_variant=target_variant,
        split = split,
        num_workers = num_workers,
    )
    node_in_dim = global_in_dim

model = spec["build_model"](node_in_dim, global_in_dim, model_hparams)
logger.info("Loaders and model built.")

2026-06-25 09:30:02,026 - __main__ - INFO - Building loaders and model for model_type=gnn...


Detected node feature dims: [17, 19, 21, 23]
Detected global feature dims: [13770, 22950]


2026-06-25 09:30:38,164 - __main__ - INFO - Loaders and model built.


In [10]:
base_dataset[0]

Data(x=[113, 23], edge_index=[2, 134], global_features=[1, 22950], y=[1], sre=[1], cid='random_q004_L020_s73412284', family='random', regime='generic_dense', n_qubits=[1], n_layers=[1], seed=[1], has_target=[1], backend='pennylane', method='fwht', representation='dense', n_bins=[1], count_rx_bin_0=[1], count_rx_bin_1=[1], count_rx_bin_2=[1], count_rx_bin_3=[1], count_rx_bin_4=[1], count_rx_bin_5=[1], count_rx_bin_6=[1], count_rx_bin_7=[1], count_rx_bin_8=[1], count_rx_bin_9=[1], count_rx_bin_10=[1], count_rx_bin_11=[1], count_rx_bin_12=[1], count_rx_bin_13=[1], count_rx_bin_14=[1], count_rx_bin_15=[1], count_rx_bin_16=[1], count_rx_bin_17=[1], count_rx_bin_18=[1], count_rx_bin_19=[1], count_rx_bin_20=[1], count_rx_bin_21=[1], count_rx_bin_22=[1], count_rx_bin_23=[1], count_rx_bin_24=[1], count_rx_bin_25=[1], count_rx_bin_26=[1], count_rx_bin_27=[1], count_rx_bin_28=[1], count_rx_bin_29=[1], count_rx_bin_30=[1], count_rx_bin_31=[1], count_rx_bin_32=[1], count_rx_bin_33=[1], count_rx_bin

In [11]:
def _amp_device_type() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"

In [12]:
huber_delta = 1.0
use_amp = True
device = _amp_device_type()
epochs=cfg.epochs
lr=cfg.lr
loss_type=cfg.loss_type
scheduler="plateau"
show_progress=cfg.show_progress
show_val_progress=cfg.show_val_progress
log_every_n_batches=cfg.log_batch_loss_every
heartbeat_secs=cfg.heartbeat
epoch_time_warning_secs=cfg.epoch_warning
weight_decay=train_hparams.get("weight_decay", 0.0)
grad_clip=train_hparams.get("grad_clip", None)
early_stopping_patience=train_hparams.get("early_stopping_patience", 30)
early_stopping_min_delta=train_hparams.get("early_stopping_min_delta", 0.0)

In [13]:
import torch
import torch.nn as nn
from typing import TYPE_CHECKING
from torch.amp.autocast_mode import autocast
from torch.amp.grad_scaler import GradScaler
from torch.optim import Adam
from tqdm import tqdm
import time

from GNN.training.utils import evaluate_loss, unpack_supervised_batch

if TYPE_CHECKING:
    from torch_geometric.loader import DataLoader

from GNN.training.train import build_loss, TrainHistory, _run_train_epoch

In [14]:
dev = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
if dev == "cuda":
    torch.backends.cudnn.benchmark = True
logger.info(f"Using device: {dev}")
model = model.to(dev)

loss_fn = build_loss(loss_type=loss_type, huber_delta=huber_delta)
optimizer = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

if scheduler == "plateau":
    lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
    )
elif scheduler == "none":
    lr_scheduler = None
else:
    msg = "scheduler must be 'none' or 'plateau'"
    raise ValueError(msg)

scaler = GradScaler(
    device=_amp_device_type(),
    enabled=(use_amp and dev.type == "cuda"),
)

history = TrainHistory(train_loss=[], val_loss=[], lr=[])

best_val_loss = float("inf")
best_state_dict = None
bad_epochs = 0

2026-06-25 09:30:41,547 - __main__ - INFO - Using device: cpu


In [15]:
history

TrainHistory(train_loss=[], val_loss=[], lr=[])

In [16]:
for epoch in range(1, epochs + 1):
        epoch_start_time = time.time()
        logger.info(f"-------- EPOCH {epoch:03d} --------")  # noqa: G004

        train_loss, train_time = _run_train_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            scaler=scaler,
            device=dev,
            use_amp=use_amp,
            grad_clip=grad_clip,
            epoch_idx=epoch,
            num_epochs=epochs,
            show_progress=show_progress,
            log_every_n_batches=log_every_n_batches,
            heartbeat_secs=heartbeat_secs,
        )

        logger.info(f"Training complete ({train_time:.1f}s) | running validation...")  # noqa: G004

        val_start_time = time.time()
        val_loss = evaluate_loss(
            model=model,
            loader=val_loader,
            device=dev,
            loss_fn=loss_fn,
            use_amp=use_amp,
            show_progress=show_val_progress,
        )
        val_time = time.time() - val_start_time

        if lr_scheduler is not None:
            lr_scheduler.step(val_loss)

        current_lr = float(optimizer.param_groups[0]["lr"])
        history.train_loss.append(float(train_loss))
        history.val_loss.append(float(val_loss))
        history.lr.append(current_lr)

        epoch_time = time.time() - epoch_start_time

        logger.info(
            f"Losses | train {train_loss:.6f} | val {val_loss:.6f} | "  # noqa: G004
            f"lr {current_lr:.2e} | time train={train_time:.1f}s "
            f"val={val_time:.1f}s total={epoch_time:.1f}s",
        )

        if epoch_time_warning_secs > 0 and epoch_time > epoch_time_warning_secs:
            logger.warning(
                f"Epoch {epoch} took {epoch_time:.1f}s "  # noqa: G004
                f"(>{epoch_time_warning_secs:.0f}s threshold).",
            )

        improved = val_loss + early_stopping_min_delta < best_val_loss
        if improved:
            best_val_loss = val_loss
            best_state_dict = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad_epochs = 0
            logger.debug(f"New best validation loss: {best_val_loss:.6f}")  # noqa: G004
        else:
            bad_epochs += 1
            logger.debug(
                f"No improvement: patience {bad_epochs}/{early_stopping_patience}",  # noqa: G004
            )
            if bad_epochs >= early_stopping_patience:
                logger.info(
                    f"Early stopping at epoch {epoch:03d} | "  # noqa: G004
                    f"best val {best_val_loss:.6f} | "
                    f"patience exhausted ({bad_epochs}/{early_stopping_patience})",
                )
                break

if best_state_dict is not None:
    model.load_state_dict(best_state_dict)

2026-06-25 09:30:42,590 - __main__ - INFO - -------- EPOCH 001 --------


Epoch 1/15:  42%|████▏     | 11/26 [01:00<01:07,  4.50s/it, loss=0.0672, graphs=1536]

2026-06-25 09:31:43,406 - GNN.training.train - INFO - [Heartbeat] Epoch 1 batch 12/26 | loss 0.067218 | elapsed 60.8s | graphs 1536


Epoch 1/15:  96%|█████████▌| 25/26 [02:01<00:04,  4.30s/it, loss=0.0404, graphs=3284]

2026-06-25 09:32:44,424 - GNN.training.train - INFO - [Heartbeat] Epoch 1 batch 26/26 | loss 0.040353 | elapsed 121.8s | graphs 3284


...

2026-06-25 09:58:51,575 - __main__ - INFO - Losses | train 0.001802 | val 0.001056 | lr 9.86e-04 | time train=115.6s val=6.1s total=121.7s
2026-06-25 09:58:51,577 - __main__ - INFO - -------- EPOCH 015 --------


Epoch 15/15:  62%|██████▏   | 16/26 [01:00<00:33,  3.32s/it, loss=0.0017, graphs=2176]

2026-06-25 09:59:52,266 - GNN.training.train - INFO - [Heartbeat] Epoch 15 batch 17/26 | loss 0.001699 | elapsed 60.7s | graphs 2176


2026-06-25 10:00:20,623 - __main__ - INFO - Training complete (89.0s) | running validation...


2026-06-25 10:00:24,602 - __main__ - INFO - Losses | train 0.001679 | val 0.001328 | lr 9.86e-04 | time train=89.0s val=3.9s total=93.0s


In [17]:
loss_fn = build_loss(cfg.loss_type, huber_delta=1.0)

test_loss = evaluate_loss(
    model,
    test_loader,
    dev,
    loss_fn,
    use_amp=True,
    show_progress=show_progress,
)
print(f"Test loss: {test_loss:.6f}")

Test loss: 0.000744


In [18]:
train_r2_score = evaluate_r2(
    model,
    train_loader,
    dev,
    show_progress=show_progress,
)

print(f"Train R^2 score: {train_r2_score:.6f}")

Train R^2 score: 0.927563


In [19]:
val_r2_score = evaluate_r2(
    model,
    val_loader,
    dev,
    show_progress=show_progress,
)
print(f"Validation R^2 score: {val_r2_score:.6f}")

Validation R^2 score: 0.906494


In [20]:
test_r2_score = evaluate_r2(
    model,
    test_loader,
    dev,
    show_progress=show_progress,
)
print(f"Test R^2 score: {test_r2_score:.6f}")

Test R^2 score: 0.925326


In [21]:
logger.info("Training complete.")

run_name = f"{model_type}_{loss_type}_{family if training_mode == 'per_family' else 'global'}"

plot_training_curves(
    history,
    title=f"{model_type.upper()} SRE regression",
    save_fig=True,
    fig_path=f"final/figures/training_curves_{run_name}.png",
)

2026-06-25 10:01:29,621 - __main__ - INFO - Training complete.
2026-06-25 10:01:30,216 - experiments.plotting - INFO - Saved training curve plot to final\figures\training_curves_gnn_huber_random_1.png


In [22]:
from GNN.training.runners import _resolve_model_save_path

save_checkpoint = True
allow_overwrite = True

model_config = {
    "node_in_dim": node_in_dim or None,
    "global_in_dim": global_in_dim,
    "hidden_dim": model_hparams.get("hidden_dim", 64),
    "dropout_rate": model_hparams.get("dropout_rate", 0.1),
}
# GNN specific params (only meaningful for gnn models)
model_config.update({
    "gnn_hidden": model_hparams.get("gnn_hidden", None),
    "gnn_heads": model_hparams.get("gnn_heads", None),
    "global_hidden": model_hparams.get("global_hidden", None),
    "reg_hidden": model_hparams.get("reg_hidden", None),
    "num_layers": model_hparams.get("num_layers", None),
})

checkpoint = {
    "model_state_dict": model.state_dict(),
    "model_type": model_type,
    "model_config": model_config,
    "target_variant": target_variant,
    "train_config": asdict(cfg),
    "train_hparams": train_hparams,
    "feature_config": {
        "global_feature_variant": cfg.global_feature_variant,
        "node_feature_backend_variant": cfg.node_feature_backend_variant,
        "all_gate_keys": getattr(base_dataset, "all_gate_keys", None),
        "family_projection": family_projection,
    },
    "final_metrics": {
        "test_loss": float(test_loss),
        "test_r2_score": float(test_r2_score),
        "train_r2_score": float(train_r2_score),
        "val_r2_score": float(val_r2_score),
    },
    "history": history,
}
model_path_root = Path("final/models")
model_path_root.mkdir(parents=True, exist_ok=True)
if save_checkpoint and model_save_path is None:
    model_save_path = _resolve_model_save_path(
        f"{model_path_root}/{run_name}.pt",
        allow_overwrite=allow_overwrite,
    )
    model_save_path = Path(str(model_save_path))
    if model_save_path.suffix == "":
        model_save_path.mkdir(parents=True, exist_ok=True)
        checkpoint_file = model_save_path / f"{family}_model_{model_type}_{training_scope}.pt"
    else:
        model_save_path.parent.mkdir(parents=True, exist_ok=True)
        checkpoint_file = model_save_path
    torch.save(checkpoint, checkpoint_file)
    logger.info(f"Saved model checkpoint to {checkpoint_file}")
elif save_checkpoint and model_save_path is not None:
    path: str = _resolve_model_save_path(
            model_save_path,
            allow_overwrite=allow_overwrite,
        )
    model_save_path = Path(path)

    model_save_path.parent.mkdir(parents=True, exist_ok=True)

    torch.save(checkpoint, model_save_path)
    logger.info(f"Saved model checkpoint to {model_save_path}")

2026-06-25 10:01:30,553 - __main__ - INFO - Saved model checkpoint to ..\notebooks\outputs\models\final\random_model_gnn_per_family_sre_density.pt


In [ ]:
from scripts import (
    generate_dataset,
    optuna_search,
    predictions,
    training,
)

In [ ]:
training_scope = "family" if training_mode == "per_family" else "global"
model_save_path = f"../notebooks/outputs/models/final/{family}_model_{model_type}_{training_mode}_{target_variant}.pt"

batch_size = 16
predictions(
    model_path=model_save_path,
    model_kind=model_type,
    training_scope=training_scope,
    target_variant=target_variant,
    loss_type=loss_type,
    model_family=family,
    dataset_root=training_data_dir,
    dataset_family=family,
    batch_size=batch_size,
    global_feature_variant="binned",
    node_feature_backend_variant=None,
    plot_n_layers=plot_layers,
    plot_n_qubits=plot_qubits,
    split_by_family=True,
    show_progress=True,
)

## Generating new datasets